# 01 — Загрузка данных
Загрузка финансовой отчётности, макро-данных, долга и лизинга в БД.

**Разделы:**
1. Настройка
2. Загрузка из Excel (IS/BS/CF + долг + лизинг)
3. Загрузка макро-факторов
4. Проверка загруженных данных
5. Диагностика препроцессора (EWA-драйверы, lease metrics, NOL)


## §1 Настройка

In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path().resolve()
ROOT = NOTEBOOK_DIR.parents[2]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

COMPANY_ID = 'us_steel'
DB_PATH    = ROOT / 'data_mart_v2.db'
DATA_DIR   = ROOT / 'companies' / COMPANY_ID / 'data'
CONFIG_PATH = ROOT / 'companies' / COMPANY_ID / 'configs' / 'excel_loader.yaml'

import logging, pandas as pd
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(message)s')

from engine.database.repository import Repository
from engine.database.schema import create_schema

# Создаём БД если нет
if not DB_PATH.exists():
    with Repository(db_path=DB_PATH) as repo:
        create_schema(repo)
    print(f'БД создана: {DB_PATH.name}')
else:
    print(f'БД существует: {DB_PATH.name}')

print(f'DATA_DIR: {DATA_DIR}')
print(f'Excel файлы: {list(DATA_DIR.glob("*.xlsx"))}')

## §2 Загрузка финансовой отчётности из Excel

In [ ]:
# Укажите путь к Excel файлу с данными компании
# Формат: шаблон из templates/excel_templates/template_UNIFIED_All_Data.xlsx
EXCEL_FILE = DATA_DIR / 'financial_data.xlsx'  # замените на ваш файл

if not EXCEL_FILE.exists():
    print(f'⚠️ Файл не найден: {EXCEL_FILE}')
    print()
    print('Доступные файлы в data/:')
    for f in DATA_DIR.glob('*.xlsx'):
        print(f'  {f.name}')
    print()
    print('Для загрузки данных:')
    print('  1. Заполните шаблон: templates/excel_templates/template_UNIFIED_All_Data.xlsx')
    print('  2. Положите в companies/{company}/data/')
    print('  3. Обновите EXCEL_FILE выше и перезапустите')
else:
    from engine.loader.excel import ExcelLoader
    import yaml
    
    # Загружаем конфиг маппинга
    if CONFIG_PATH.exists():
        with open(CONFIG_PATH) as f:
            loader_cfg = yaml.safe_load(f)
    else:
        loader_cfg = {}
    
    with Repository(db_path=DB_PATH) as repo:
        loader = ExcelLoader(
            company_id=COMPANY_ID,
            repo=repo,
            config=loader_cfg,
        )
        result = loader.load(EXCEL_FILE)
        print(f'Загружено: {result.rows_written} строк')
        print(f'IS: {result.is_rows} строк')
        print(f'BS: {result.bs_rows} строк')
        print(f'CF: {result.cf_rows} строк')
        if result.errors:
            for e in result.errors[:5]:
                print(f'  ❌ {e}')

## §3 Загрузка макро-факторов

In [ ]:
# Вариант А: загрузка из CSV файлов
MACRO_DIR = DATA_DIR / 'macro'
csv_files = list(MACRO_DIR.glob('*.csv')) if MACRO_DIR.exists() else []
print(f'CSV файлов найдено: {len(csv_files)}')

if csv_files:
    with Repository(db_path=DB_PATH) as repo:
        for csv_path in csv_files:
            factor_name = csv_path.stem  # имя файла = имя фактора
            try:
                df = pd.read_csv(csv_path)
                # Ожидаем колонки: year, value
                if 'year' in df.columns and 'value' in df.columns:
                    data = dict(zip(df['year'].astype(int), df['value'].astype(float)))
                    n = repo.upsert_macro_factors({factor_name: data}, scope='global', source='csv')
                    print(f'  ✅ {factor_name}: {n} точек')
                else:
                    print(f'  ⚠️ {csv_path.name}: нужны колонки year, value')
            except Exception as e:
                print(f'  ❌ {csv_path.name}: {e}')
else:
    print('  Положите CSV файлы в companies/{company}/data/macro/')
    print('  Формат: year,value (одна строка = один год)')
    print()
    print('Текущие факторы в БД:')
    with Repository(db_path=DB_PATH) as repo:
        rows = repo.query('SELECT DISTINCT factor_name FROM macro_factors ORDER BY factor_name')
        for r in rows:
            series = repo.get_macro_factor(r['factor_name'])
            last_yr = max(series.keys()) if series else 'N/A'
            print(f'  {r["factor_name"]}: {len(series)} лет, last={last_yr}')

## §4 Проверка загруженных данных

In [ ]:
with Repository(db_path=DB_PATH) as repo:
    # Проверяем IS
    print('=== Income Statement ===')
    rows_is = repo.query(
        """SELECT p.year, COUNT(*) as n_metrics
        FROM history_is h JOIN periods p ON h.period_id=p.period_id
        WHERE h.company_id=? AND p.is_annual=1
        GROUP BY p.year ORDER BY p.year""",
        (COMPANY_ID,)
    )
    if rows_is:
        for r in rows_is[-5:]:
            print(f'  {r["year"]}: {r["n_metrics"]} метрик')
    else:
        print('  Нет данных')
    
    print()
    print('=== Balance Sheet ===')
    rows_bs = repo.query(
        """SELECT p.year, COUNT(*) as n_metrics
        FROM history_bs h JOIN periods p ON h.period_id=p.period_id
        WHERE h.company_id=? AND p.is_annual=1
        GROUP BY p.year ORDER BY p.year""",
        (COMPANY_ID,)
    )
    if rows_bs:
        for r in rows_bs[-5:]:
            print(f'  {r["year"]}: {r["n_metrics"]} метрик')
    else:
        print('  Нет данных')
    
    print()
    print('=== Ключевые метрики 2024 ===')
    is_2024 = repo.get_history_year(COMPANY_ID, 'IS', 2024)
    bs_2024 = repo.get_history_year(COMPANY_ID, 'BS', 2024)
    
    key_is = ['revenue','cogs','gross_profit','ebitda','ebit','net_income']
    key_bs = ['cash','total_assets','total_equity','short_term_debt','long_term_debt']
    
    print('IS:')
    for k in key_is:
        v = is_2024.get(k)
        if v: print(f'  {k:<25} {v/1e6:>10.0f} M')
    print('BS:')
    for k in key_bs:
        v = bs_2024.get(k)
        if v: print(f'  {k:<25} {v/1e6:>10.0f} M')

## §5 Диагностика качества данных

In [ ]:
from engine.preprocessor.core import ModelPreprocessor

with Repository(db_path=DB_PATH) as repo:
    pp = ModelPreprocessor(COMPANY_ID, repo)
    pp_result = pp.run()
    
    print(f'Препроцессор: {"OK" if pp_result.success else "FAIL"}')
    print(f'  Групп:   {len(pp_result.groups_computed)}')
    print(f'  Метрик:  {pp_result.metrics_written}')
    print()
    
    # Ключевые метрики
    margins = repo.get_preprocess(COMPANY_ID, 'margin_ratios') or {}
    wc = repo.get_preprocess(COMPANY_ID, 'wc_days') or {}
    lease = repo.get_preprocess(COMPANY_ID, 'lease') or {}
    
    print('Ключевые EWA-метрики препроцессора:')
    for group, data in [('margin_ratios', margins), ('wc_days', wc)]:
        if data:
            print(f'  [{group}]')
            for k, v in sorted(data.items()):
                if k.endswith('_recommended') and v is not None:
                    print(f'    {k}: {v:.4f}')
    print()
    
    print('Lease метрики (из preprocessor):')
    lease_keys = ['op_lease_decay_rate', 'op_lease_new_leases', 'op_lease_cash_payment',
                  'fin_lease_principal_rate', 'fin_lease_amort_rate', 'fin_lease_interest_rate',
                  'fin_lease_new_leases']
    for k in lease_keys:
        rec = lease.get(k + '_recommended')
        if rec is not None:
            unit = 'M' if 'new_leases' in k or 'cash' in k or 'payment' in k else ''
            val_str = f'{rec*1e-6:.1f}M' if unit == 'M' else f'{rec:.4f}'
            print(f'  {k}: {val_str}')
    print()
    
    print('NOL (из project.yaml):')
    from engine.model.loader import ModelInputLoader
    from pathlib import Path as P
    proj_path = P(str(DB_PATH).replace('data_mart_v2.db', '')) / 'companies/us_steel/configs/project.yaml'
    with Repository(db_path=DB_PATH) as repo2:
        ldr = ModelInputLoader(COMPANY_ID, repo2, proj_path, 'base')
        _, cfg = ldr.load()
    print(f'  nol_enabled:         {cfg.nol_enabled}')
    print(f'  nol_opening_balance: ${cfg.nol_opening_balance/1e6:.0f}M')
    print(f'  nol_limit_pct:       {cfg.nol_limit_pct}')
